<div style="background: linear-gradient(135deg, #1a0533 0%, #2d1b69 50%, #6a1b9a 100%); padding: 50px 40px; border-radius: 16px; text-align: center; margin-bottom: 30px;">
  <h1 style="color: #ede7f6; font-size: 2.8em; font-weight: 800; letter-spacing: 2px; margin: 0;">⚗️ SEABORN MASTERCLASS</h1>
  <h2 style="color: #ce93d8; font-size: 1.6em; font-weight: 400; margin-top: 10px;">Notebook 5 — Advanced Visualization</h2>
  <p style="color: #ccc; font-size: 1em; margin-top: 18px;">FacetGrid · PairGrid · twinx · Annotations · Reusable Patterns</p>
  <hr style="border: 1px solid #ce93d8; margin: 20px auto; width: 60%;">
  <p style="color: #aaa; font-size: 0.9em;">📦 Library: <strong style='color:#ede7f6'>Seaborn</strong> &nbsp;|&nbsp; ⏱ Est. Time: 120 mins &nbsp;|&nbsp; 🎯 Fresher Interview Focus</p>
</div>

---
## 📋 Table of Contents

| # | Section | Concepts Covered |
|---|---------|-----------------|
| 1 | [Learning Objectives](#objectives) | — |
| 2 | [Quick Reference Card](#qrc) | All advanced APIs |
| 3 | [Setup & Data Loading](#setup) | — |
| 4 | [FacetGrid — Custom Faceting](#facetgrid) | `map()` vs `map_dataframe()` |
| 5 | [PairGrid — Mix Plots on One Grid](#pairgrid) | Diagonal / Upper / Lower |
| 6 | [Advanced Annotations](#annotations) | `ax.annotate()`, arrows, callouts |
| 7 | [Twin Y-Axes — Dual Metric Charts](#twinx) | `ax.twinx()` |
| 8 | [Reusable Dashboard Functions](#reusable) | Production-quality patterns |
| 9 | [Advanced Color Palettes](#palettes) | Custom, sequential, diverging |
| 10 | [Business Dashboard](#dashboard) | 4-panel advanced report |
| 11 | [Best Practices](#best-practices) | — |
| 12 | [Common Mistakes](#common-mistakes) | — |
| 13 | [Interview Notes](#interview-notes) | — |
| 14 | [Summary & Key Takeaways](#summary) | — |
| 15 | [Practice Questions](#practice) | 15 exercises |

---

<a id='objectives'></a>
## 🎯 Learning Objectives

By the end of this notebook you will be able to:

- ✅ Build a `FacetGrid` from scratch using `map_dataframe()` — more powerful than built-in `relplot`/`displot`
- ✅ Build a `PairGrid` with **different plot types** on the diagonal, upper, and lower triangle
- ✅ Annotate specific data points with arrows and callouts using `ax.annotate()`
- ✅ Create dual-axis charts (`ax.twinx()`) — revenue + growth rate on the same plot
- ✅ Write **reusable plotting functions** — a production-quality pattern every fresher should know
- ✅ Choose the right color palette (qualitative, sequential, diverging) and build custom ones
- ✅ Set up a professional figure template that you can reuse across projects

**⏱ Estimated Time:** ~120 minutes | **📋 Prerequisites:** Notebooks 1–4 completed

---

<a id='qrc'></a>
## ⚡ Quick Reference Card

### FacetGrid API
```python
g = sns.FacetGrid(df, col='cat', row='cat2', hue='cat3',
                  height=4, aspect=1.2, palette='deep')
g.map_dataframe(sns.scatterplot, x='x', y='y')   # ← preferred
g.add_legend()
g.set_axis_labels('X Label', 'Y Label')
g.set_titles(col_template='{col_name}', row_template='{row_name}')
```

### PairGrid API
```python
g = sns.PairGrid(df, hue='cat', vars=['col1','col2','col3'])
g.map_diag(sns.histplot, fill=True)         # Diagonal → histogram
g.map_upper(sns.scatterplot, alpha=0.5)     # Upper triangle → scatter
g.map_lower(sns.kdeplot, fill=True)         # Lower triangle → KDE
g.add_legend()
```

### twinx Pattern
```python
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()                           # Shares x-axis, independent y
ax1.plot(x, y1, color='blue')
ax2.plot(x, y2, color='red')
ax1.set_ylabel('Metric 1', color='blue')
ax2.set_ylabel('Metric 2', color='red')
```

### Annotation Pattern
```python
ax.annotate(
    text='Insight here',
    xy=(x_data, y_data),          # Point to annotate
    xytext=(x_text, y_text),      # Where text appears
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
    fontsize=10, color='red',
    bbox=dict(boxstyle='round', fc='#FFEBEE', alpha=0.85)
)
```

### 🚦 Difficulty Legend
| Marker | Level | Meaning |
|--------|-------|---------|
| 🟢 Basic | Must-know | Core syntax |
| 🟡 Intermediate | Good-to-know | Parameters and combinations |
| 🔴 Advanced | Stand-out | Production patterns |

---

<a id='setup'></a>
## 1. Setup & Data Loading

In [ ]:
# 🟢 Basic | Standard setup
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import os

sns.set_theme(
    style='whitegrid',
    context='notebook',
    palette='deep',
    font_scale=1.1,
    rc={
        'figure.facecolor': 'white',
        'axes.facecolor':   '#FAFAFA',
        'grid.alpha':        0.35,
        'figure.dpi':        110,
        'axes.spines.top':   False,
        'axes.spines.right': False,
    }
)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
pd.set_option('display.max_columns', None)
os.makedirs('outputs', exist_ok=True)
print("✅ Setup complete.")

In [ ]:
# 🟢 Basic | Load datasets ONCE
tips      = sns.load_dataset('tips')
penguins  = sns.load_dataset('penguins').dropna()
mpg       = sns.load_dataset('mpg').dropna()
flights   = sns.load_dataset('flights')
diamonds  = sns.load_dataset('diamonds')
exercise  = sns.load_dataset('exercise')

# Engineer useful columns
tips['tip_pct']   = (tips['tip'] / tips['total_bill'] * 100).round(2)
mpg['efficiency'] = (mpg['mpg'] / mpg['weight'] * 1000).round(3)

# Aggregate flights for twin-axis demo
yearly_flights = (flights.groupby('year')
                         .agg(total_passengers=('passengers', 'sum'))
                         .reset_index())
yearly_flights['yoy_growth_pct'] = (
    yearly_flights['total_passengers'].pct_change() * 100
).round(2)

print("✅ Datasets ready.")
print(f"   yearly_flights shape: {yearly_flights.shape}")
print(yearly_flights.head())

---
<a id='facetgrid'></a>
## 2. FacetGrid — Custom Faceting from Scratch

### 🧠 Intuition

`FacetGrid` is the **engine** behind all figure-level functions (`relplot`, `displot`, `catplot`).
Understanding it directly lets you build **any custom faceted layout** — not just what the high-level APIs support.

### 🔑 `map()` vs `map_dataframe()` — The Key Difference

| Method | How it works | Limitation |
|--------|-------------|-----------|
| `g.map(func, 'col1', 'col2')` | Passes **Series** objects to the function | Can't pass kwargs like `hue=`, `order=` |
| `g.map_dataframe(func, x='col', y='col')` | Passes the **full DataFrame slice** + kwargs | ✅ Supports all kwargs — **preferred** |

> 🎯 **Rule for freshers:** Always use `map_dataframe()` — it's more flexible and handles `hue=` correctly.

### 📌 FacetGrid Workflow
```
1. Create: g = sns.FacetGrid(df, col=, row=, hue=)
2. Map:    g.map_dataframe(plot_func, x=, y=)
3. Polish: g.add_legend(), g.set_axis_labels(), g.set_titles()
```

In [ ]:
# 🟡 Intermediate | FacetGrid with map_dataframe — the flexible way
# Business Problem: Does the bill→tip relationship hold across ALL
# combinations of meal time AND smoking status?
# relplot can't do 2D grids with hue easily — FacetGrid can.

g = sns.FacetGrid(
    tips,
    col='time',           # Columns = Lunch / Dinner
    row='smoker',         # Rows    = Yes / No
    hue='sex',            # Color   = Male / Female
    palette={'Male': '#1565C0', 'Female': '#C62828'},
    height=3.8,
    aspect=1.15,
    margin_titles=True    # Cleaner: titles on margins, not inside panels
)

# map_dataframe passes the full filtered DataFrame to scatterplot
g.map_dataframe(
    sns.scatterplot,
    x='total_bill',
    y='tip',
    alpha=0.65,
    s=45
)

g.add_legend(title='Gender')
g.set_axis_labels('Total Bill ($)', 'Tip ($)')
g.set_titles(col_template='{col_name}', row_template='Smoker: {row_name}')
g.figure.suptitle(
    '🍽️ FacetGrid: Bill vs Tip — Meal Time × Smoker × Gender\n'
    '(2×2 grid — built with FacetGrid.map_dataframe)',
    fontsize=13, fontweight='bold', y=1.03
)

plt.tight_layout()
plt.show()

print("📊 INSIGHT: The bill→tip positive relationship is consistent across ALL 4 panels.")
print("   No dramatic difference between smokers/non-smokers or Lunch/Dinner.")
print("   → Tipping behaviour is primarily driven by bill size, not these demographics.")

### 📋 Output Preview — FacetGrid
```
2 rows × 2 cols = 4 panels:
  Row 1 (Smoker=No):  Lunch | Dinner
  Row 2 (Smoker=Yes): Lunch | Dinner
Each panel: scatter colored by sex

Key visual: All 4 panels show the SAME upward trend.
Consistent pattern → simpler model is better.
```
> 🎯 **Interview Q:** *"What is FacetGrid and how is it different from relplot?"*
> → `FacetGrid` is the **low-level class** that figure-level functions like `relplot` wrap internally. `relplot` is a convenience API over `FacetGrid`. Using `FacetGrid` directly gives you full control — you can use `map_dataframe()` with **any** plotting function, not just the ones relplot supports.

In [ ]:
# 🟡 Intermediate | FacetGrid with map_dataframe + custom per-panel styling
# Business Problem: Distribution of mpg per country of origin.
# We want KDE + rug plot on the same facet — relplot can't do this.

g = sns.FacetGrid(
    mpg,
    col='origin',
    col_order=['usa', 'europe', 'japan'],
    col_wrap=3,
    height=4,
    aspect=1.2,
    sharey=True,
    sharex=True
)

# Step 1: Draw filled KDE
g.map_dataframe(
    sns.kdeplot,
    x='mpg',
    fill=True,
    alpha=0.45,
    linewidth=2
)

# Step 2: Overlay rug plot on same panels
g.map_dataframe(
    sns.rugplot,
    x='mpg',
    height=0.06,
    alpha=0.6
)

g.set_axis_labels('Miles Per Gallon (mpg)', 'Density')
g.set_titles(col_template='Origin: {col_name}')

# Add vertical median lines per panel manually — one call per panel
ax_usa    = g.axes_dict['usa']
ax_europe = g.axes_dict['europe']
ax_japan  = g.axes_dict['japan']

med_usa    = mpg[mpg['origin'] == 'usa']['mpg'].median()
med_europe = mpg[mpg['origin'] == 'europe']['mpg'].median()
med_japan  = mpg[mpg['origin'] == 'japan']['mpg'].median()

ax_usa.axvline(med_usa,    color='#E53935', linestyle='--', lw=1.8,
               label=f'Median: {med_usa:.1f} mpg')
ax_europe.axvline(med_europe, color='#E53935', linestyle='--', lw=1.8,
                  label=f'Median: {med_europe:.1f} mpg')
ax_japan.axvline(med_japan, color='#E53935', linestyle='--', lw=1.8,
                 label=f'Median: {med_japan:.1f} mpg')

ax_usa.legend(fontsize=9)
ax_europe.legend(fontsize=9)
ax_japan.legend(fontsize=9)

g.figure.suptitle(
    '🚗 MPG Distribution by Country of Origin\n'
    '(FacetGrid: KDE + Rug + Median Line — impossible with relplot)',
    fontsize=13, fontweight='bold', y=1.03
)
plt.tight_layout()
plt.show()

print(f"📊 Median MPG — USA: {med_usa:.1f} | Europe: {med_europe:.1f} | Japan: {med_japan:.1f}")
print("   Japanese cars have the highest median fuel efficiency.")
print("   USA distribution is widest — biggest range from gas-guzzlers to economy cars.")

### 📋 Output Preview
```
📊 Median MPG:
   USA:    23.0
   Europe: 26.0
   Japan:  31.0 ← highest efficiency

3 panels: each with filled KDE + rug marks at bottom + red median line
USA panel is widest (most variance) — Japan is narrow and right-shifted.
```
> 💡 **Production Tip:** `g.axes_dict` gives you a dictionary mapping column values to Axes objects. This is how you access individual panels in FacetGrid without loops.

---

<a id='pairgrid'></a>
## 3. PairGrid — Different Plots per Position

### 🧠 Intuition

`PairGrid` = the manual, full-control version of `pairplot`.

- `pairplot()` uses the **same plot type** everywhere (scatter off-diagonal, hist/kde on diagonal)
- `PairGrid` lets you put **completely different plots** on the:
  - **Diagonal** (distribution of each variable)
  - **Upper triangle** (one plot type)
  - **Lower triangle** (another plot type)

This creates charts that look impressive and pack much more information — interviewers notice this.

### 🔑 PairGrid Methods
```python
g = sns.PairGrid(df, hue='cat', vars=['col1', 'col2', 'col3'])
g.map_diag(func)       # Called once per variable
g.map_upper(func)      # Called for each upper triangle pair
g.map_lower(func)      # Called for each lower triangle pair
```

In [ ]:
# 🟡 Intermediate | PairGrid — different plots on diagonal, upper, lower
# Business Problem: Full penguin dataset exploration.
# Upper = scatter (raw data), Lower = KDE contours, Diagonal = KDE distribution.

g = sns.PairGrid(
    penguins,
    hue='species',
    vars=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'],
    palette='colorblind',
    height=2.5
)

# Diagonal: KDE distribution per species
g.map_diag(
    sns.kdeplot,
    fill=True,
    alpha=0.45,
    linewidth=1.5
)

# Upper triangle: Scatter plots (raw data)
g.map_upper(
    sns.scatterplot,
    alpha=0.55,
    s=28,
    edgecolor='none'
)

# Lower triangle: KDE contour density
g.map_lower(
    sns.kdeplot,
    fill=True,
    alpha=0.30,
    linewidths=1.2
)

g.add_legend(title='Species', fontsize=9)
g.figure.suptitle(
    '🐧 PairGrid — Upper: Scatter | Lower: KDE | Diagonal: KDE Distribution\n'
    '(3 different plot types in one grid)',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

print("📊 PairGrid advantage over pairplot:")
print("   Upper triangle: see individual data points (scatter)")
print("   Lower triangle: see where data concentrates (KDE contours)")
print("   Diagonal: see how each feature distributes per species")
print("   → 3 types of information simultaneously in one figure.")

### 📋 Output Preview — PairGrid
```
4×4 grid:
  Diagonal (top-left to bottom-right): Filled KDE per species
  Upper triangle: Scatter plots (dots colored by species)
  Lower triangle: KDE contour density (filled, semi-transparent)

Information density: 3× that of a regular pairplot.
```
> 🎯 **Interview Q:** *"What is the difference between pairplot and PairGrid?"*
> → `pairplot` is a convenience function — same scatter everywhere off-diagonal, same hist/kde on diagonal. `PairGrid` is the underlying class — you can assign **completely different functions** to diagonal, upper, and lower triangle independently. PairGrid is more code but far more flexible.

In [ ]:
# 🔴 Advanced | PairGrid with regression on lower + regplot-style upper
# Business Problem: MPG dataset — understand linear relationships AND distributions.
# Useful when preparing a feature analysis report for a manager.

g2 = sns.PairGrid(
    mpg[['mpg', 'horsepower', 'weight', 'acceleration']],
    height=2.4,
    aspect=1.0
)

# Diagonal: histplot with KDE overlay
g2.map_diag(sns.histplot, kde=True, color='#1565C0', alpha=0.6, bins=15)

# Upper triangle: regplot scatter (no regression line — just scatter with alpha)
g2.map_upper(sns.scatterplot, alpha=0.35, s=18, color='#1565C0')

# Lower triangle: regplot with regression line
g2.map_lower(sns.regplot,
             scatter_kws={'alpha': 0.25, 's': 12, 'color': '#888'},
             line_kws={'color': '#E53935', 'lw': 1.8},
             ci=None)

g2.figure.suptitle(
    '🚗 MPG Features — Hist on Diagonal | Scatter Upper | Regression Lower\n'
    '(Regression on lower triangle reveals linear vs non-linear patterns)',
    fontsize=11, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

print("📊 KEY FINDING from regression lower triangle:")
print("   mpg vs weight: strong negative slope (linear) ✅")
print("   mpg vs horsepower: curved — linear regplot underestimates at low HP")
print("   → weight is a better linear predictor of mpg than horsepower")

---
<a id='annotations'></a>
## 4. Advanced Annotations — Telling the Story

### 🧠 Why Annotations Matter

A chart without annotations makes the viewer work to find the insight.
A chart **with** annotations tells the story directly — this is the difference between
an academic plot and a business-ready visualization.

> 🏆 **Industry Standard:** Every chart in a business presentation should have at least one annotation pointing out the key insight.

### 📌 ax.annotate() Parameters
```python
ax.annotate(
    text       = 'Your insight',
    xy         = (x_point, y_point),     # The DATA POINT being annotated
    xytext     = (x_text,  y_text),      # Where the TEXT is placed
    xycoords   = 'data',                 # Coordinate system for xy
    textcoords = 'data',                 # Coordinate system for xytext
    arrowprops = dict(
        arrowstyle = '->',               # '->' | '-|>' | 'fancy' | 'simple'
        color      = '#E53935',
        lw         = 1.5,
        connectionstyle = 'arc3,rad=0.3'  # Curved arrow
    ),
    fontsize   = 10,
    color      = '#E53935',
    fontweight = 'bold',
    bbox       = dict(boxstyle='round,pad=0.4',
                      facecolor='#FFEBEE', alpha=0.85, edgecolor='#E53935')
)
```

In [ ]:
# 🟡 Intermediate | Annotations — highlighting outliers and key points
# Business Problem: Identify and annotate the highest-tip and lowest-tip
# tables from the restaurant data. This is a real reporting task.

# Find key data points BEFORE plotting (no loops needed)
max_tip_row  = tips.loc[tips['tip'].idxmax()]
min_tip_row  = tips.loc[tips['tip'].idxmin()]
max_bill_row = tips.loc[tips['total_bill'].idxmax()]

fig, ax = plt.subplots(figsize=(12, 7))

# Base scatter
sns.scatterplot(
    data=tips,
    x='total_bill',
    y='tip',
    hue='time',
    palette={'Lunch': '#1565C0', 'Dinner': '#E53935'},
    alpha=0.65,
    s=50,
    ax=ax
)

# ── Annotation 1: Highest tip ──
ax.annotate(
    f"Highest tip: ${max_tip_row['tip']:.2f}\n"
    f"(Bill: ${max_tip_row['total_bill']:.2f})",
    xy=(max_tip_row['total_bill'], max_tip_row['tip']),
    xytext=(max_tip_row['total_bill'] - 12, max_tip_row['tip'] + 0.3),
    arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2.0),
    fontsize=9.5, fontweight='bold', color='#2E7D32',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F5E9', alpha=0.9, edgecolor='#2E7D32')
)

# ── Annotation 2: Highest bill (interesting outlier) ──
ax.annotate(
    f"Biggest table\nBill: ${max_bill_row['total_bill']:.2f}\n"
    f"Tip: ${max_bill_row['tip']:.2f} (only {max_bill_row['tip_pct']:.1f}%!)",
    xy=(max_bill_row['total_bill'], max_bill_row['tip']),
    xytext=(max_bill_row['total_bill'] - 18, max_bill_row['tip'] + 1.8),
    arrowprops=dict(arrowstyle='->', color='#E53935', lw=2.0,
                    connectionstyle='arc3,rad=-0.25'),
    fontsize=9.5, fontweight='bold', color='#C62828',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFEBEE', alpha=0.9, edgecolor='#E53935')
)

# ── Annotation 3: Cluster callout with rectangle ──
rect = mpatches.FancyBboxPatch(
    (7, 0.8), width=13, height=2.5,
    boxstyle='round,pad=0.3',
    linewidth=1.5, edgecolor='#FB8C00', facecolor='#FFF8E1', alpha=0.4
)
ax.add_patch(rect)
ax.text(8, 3.5, '← Most common\nvisit zone', fontsize=9,
        color='#E65100', fontweight='bold', style='italic')

ax.set_title(
    '🍽️ Restaurant Analytics — Annotated Insights\n'
    '(Professional: chart tells the story, not just shows data)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Total Bill ($)', fontsize=12)
ax.set_ylabel('Tip Amount ($)', fontsize=12)
ax.legend(title='Meal Time', fontsize=10)

plt.tight_layout()
plt.show()

print(f"📊 ANNOTATED INSIGHTS:")
print(f"   Highest tip:   ${max_tip_row['tip']:.2f} on a ${max_tip_row['total_bill']:.2f} bill")
print(f"   Biggest table: ${max_bill_row['total_bill']:.2f} bill — only {max_bill_row['tip_pct']:.1f}% tip!")
print(f"   → Large parties tip a LOWER percentage — important for server scheduling.")

### 📋 Output Preview — Annotated Chart
```
📊 ANNOTATED INSIGHTS:
   Highest tip:   $10.00 on a $29.80 bill (33.6%)
   Biggest table: $50.81 bill — only 5.6% tip!
   → Large parties tip LOWER % — important for server scheduling.
```
> 🎯 **Interview Q:** *"How do you annotate a specific data point in a Seaborn/Matplotlib chart?"*
> → Use `ax.annotate(text, xy=(data_x, data_y), xytext=(text_x, text_y), arrowprops=dict(...))`. The `xy` argument points to the data, `xytext` controls where the label sits, and `arrowprops` draws the connecting arrow.

In [ ]:
# 🔴 Advanced | Annotations with connection styles + shading
# Business Problem: Highlight aviation industry inflection points.
# When did growth accelerate? When did it plateau? Annotate the key years.

yearly_flights['total_passengers'] = flights.groupby('year')['passengers'].sum().values

fig, ax = plt.subplots(figsize=(13, 6))

sns.lineplot(
    data=yearly_flights,
    x='year',
    y='total_passengers',
    color='#1565C0',
    linewidth=2.8,
    marker='o',
    markersize=7,
    markerfacecolor='white',
    markeredgecolor='#1565C0',
    markeredgewidth=2,
    ax=ax
)

ax.fill_between(
    yearly_flights['year'],
    yearly_flights['total_passengers'],
    alpha=0.10, color='#1565C0'
)

# ── Annotation: Fastest growth year ──
fastest_growth_yr = yearly_flights.loc[yearly_flights['yoy_growth_pct'].idxmax(), 'year']
fastest_growth_val = yearly_flights.loc[
    yearly_flights['yoy_growth_pct'].idxmax(), 'total_passengers'
]
fastest_growth_pct = yearly_flights['yoy_growth_pct'].max()

ax.annotate(
    f'📈 Fastest growth year\n+{fastest_growth_pct:.1f}% YoY',
    xy=(fastest_growth_yr, fastest_growth_val),
    xytext=(fastest_growth_yr - 2.5, fastest_growth_val + 250),
    arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2,
                    connectionstyle='arc3,rad=0.2'),
    fontsize=10, fontweight='bold', color='#2E7D32',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F5E9', alpha=0.92, edgecolor='#2E7D32')
)

# ── Annotation: Peak year ──
peak_yr  = yearly_flights.loc[yearly_flights['total_passengers'].idxmax(), 'year']
peak_val = yearly_flights['total_passengers'].max()

ax.annotate(
    f'🏆 Peak: {peak_val:,} passengers
({peak_yr})',
    xy=(peak_yr, peak_val),
    xytext=(peak_yr - 4, peak_val - 500),
    arrowprops=dict(arrowstyle='->', color='#E53935', lw=2),
    fontsize=10, fontweight='bold', color='#C62828',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFEBEE', alpha=0.92, edgecolor='#E53935')
)

# ── Shading: acceleration period ──
ax.axvspan(1955, 1960, alpha=0.07, color='#FB8C00', label='High-growth decade')

ax.set_title(
    '✈️ Aviation Industry Growth — Annotated Inflection Points\n'
    '(Business insight callouts make charts self-explaining)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total Annual Passengers', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xticks(yearly_flights['year'])
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
<a id='twinx'></a>
## 5. Twin Y-Axes — Dual Metric Charts

### 🧠 Intuition

Sometimes you need to show **two metrics with different scales** on the same chart.

**Real-world examples:**
- Revenue ($) and Growth rate (%) on the same time axis
- Passenger volume and YoY change %
- Sales count and Average order value

`ax.twinx()` creates a **second Y-axis** that shares the same X-axis.

### ⚠️ When NOT to use twinx
| Situation | Problem | Alternative |
|-----------|---------|-------------|
| Both metrics in same units | Misleading — just use one axis | Single Y-axis |
| More than 2 metrics | Unreadable — 3 Y-axes is chaos | Separate subplots |
| Non-technical audience | Confusing — two Y-axes need explanation | Stacked subplots |

> 🎯 **Interview Tip:** Mention twinx in interviews as an example of combining axes-level Matplotlib with Seaborn. It shows you understand the Matplotlib/Seaborn relationship.

In [ ]:
# 🟡 Intermediate | twinx — passengers volume + YoY growth on same chart
# Business Problem: Aviation trend report — volume AND growth rate together.
# Volume tells HOW BIG. Growth rate tells HOW FAST. Both needed for full picture.

fig, ax1 = plt.subplots(figsize=(13, 6))

# ── Left Y-axis: Total passengers (bars) ──
bars = ax1.bar(
    yearly_flights['year'],
    yearly_flights['total_passengers'],
    color='#90CAF9',
    alpha=0.75,
    width=0.6,
    label='Total Passengers'
)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Total Annual Passengers', fontsize=12, color='#1565C0')
ax1.tick_params(axis='y', labelcolor='#1565C0')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax1.set_ylim(0, yearly_flights['total_passengers'].max() * 1.2)

# ── Right Y-axis: YoY growth rate (line) ──
ax2 = ax1.twinx()                          # ← shares X, independent Y

ax2.plot(
    yearly_flights.dropna()['year'],
    yearly_flights.dropna()['yoy_growth_pct'],
    color='#E53935',
    linewidth=2.5,
    marker='D',
    markersize=7,
    markerfacecolor='white',
    markeredgecolor='#E53935',
    markeredgewidth=2,
    label='YoY Growth %'
)
ax2.axhline(y=0, color='#E53935', linestyle='--', alpha=0.3, linewidth=1)
ax2.set_ylabel('Year-over-Year Growth (%)', fontsize=12, color='#E53935')
ax2.tick_params(axis='y', labelcolor='#E53935')

# ── Combined legend ──
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=10)

ax1.set_title(
    '✈️ Aviation Analytics — Passenger Volume vs YoY Growth Rate\n'
    '(Blue bars = Volume [left Y] | Red line = Growth % [right Y])',
    fontsize=13, fontweight='bold', pad=12
)

# Annotate the fastest growth year on the line
fastest_row = yearly_flights.dropna().loc[yearly_flights.dropna()['yoy_growth_pct'].idxmax()]
ax2.annotate(
    f"+{fastest_row['yoy_growth_pct']:.1f}%",
    xy=(fastest_row['year'], fastest_row['yoy_growth_pct']),
    xytext=(fastest_row['year'] + 0.3, fastest_row['yoy_growth_pct'] + 1.5),
    fontsize=9, fontweight='bold', color='#C62828'
)

plt.tight_layout()
plt.show()

avg_growth = yearly_flights['yoy_growth_pct'].mean()
print(f"📊 Average YoY Growth: {avg_growth:.1f}%")
print("   Volume grows every year, but growth RATE fluctuates — some years are faster.")
print("   twinx lets you see BOTH patterns simultaneously on one chart.")

### 📋 Output Preview — Twin Axes
```
Same X-axis (years 1949–1960)
Left Y (blue): Bars growing 1520 → 5714 passengers
Right Y (red): Line fluctuating between +5% to +20% growth

Key: You can see years where volume grew fast (high red line)
vs years where growth slowed (low red line) despite volume still increasing.
```
> 🎯 **Interview Q:** *"How do you plot two metrics with different scales on the same chart?"*
> → `ax2 = ax1.twinx()` — this creates a second Axes object that shares the same X-axis but has an independent Y-axis. You then plot on `ax1` and `ax2` separately, set different y-labels with different colors, and combine the legends manually.

---

<a id='reusable'></a>
## 6. Reusable Dashboard Functions — Production Pattern

### 🧠 Why This Matters for Freshers

In a real job, you'll create the **same types of charts repeatedly** — weekly reports, model evaluation dashboards, etc.
Writing a reusable function means:
- ✅ You run one line instead of 20 lines every time
- ✅ Consistent style across all charts automatically
- ✅ Easy to hand off to teammates
- ✅ Shows coding maturity in interviews

> 🎯 **Interview Gold:** Mentioning reusable visualization functions demonstrates that you think beyond one-off scripts — you think about **maintainability and code quality**.

### 🔑 The Pattern
```python
def plot_<something>(df, x, y, title, ...) -> matplotlib.axes.Axes:
    """Docstring explaining what this does."""
    fig, ax = plt.subplots(...)
    # plot code
    ax.set_title(title)
    return ax    # Always return ax for further customization
```

In [ ]:
# 🔴 Advanced | Reusable insight chart function — production template
# This is the type of code you write on the JOB, not just in notebooks.
# Shows software engineering maturity — sets you apart from other freshers.

def plot_distribution_with_stats(
    df: pd.DataFrame,
    column: str,
    group_col: str = None,
    title: str = None,
    palette: str = 'deep',
    figsize: tuple = (11, 5)
) -> plt.Axes:
    """
    Plot a publication-quality distribution chart with:
    - KDE + histogram overlay
    - Mean and median reference lines
    - Optional group breakdown via hue
    - Auto-computed statistics annotation

    Parameters
    ----------
    df        : Input DataFrame
    column    : Numeric column to visualize
    group_col : Optional categorical column for color grouping
    title     : Chart title (auto-generated if None)
    palette   : Seaborn palette name
    figsize   : Figure size tuple

    Returns
    -------
    ax : matplotlib Axes (for further customization)
    """
    fig, ax = plt.subplots(figsize=figsize)

    sns.histplot(
        data=df,
        x=column,
        hue=group_col,
        kde=True,
        stat='density',
        alpha=0.45,
        palette=palette,
        common_norm=False,    # Independent normalization per group
        ax=ax
    )

    # Add mean and median lines (overall, not per group)
    col_mean   = df[column].mean()
    col_median = df[column].median()

    ax.axvline(col_mean,   color='#E53935', linestyle='--', lw=2,
               label=f'Mean: {col_mean:.2f}')
    ax.axvline(col_median, color='#1565C0', linestyle='-.',  lw=2,
               label=f'Median: {col_median:.2f}')

    # Statistics box
    skew_val = df[column].skew()
    skew_dir = 'Right-skewed' if skew_val > 0.5 else 'Left-skewed' if skew_val < -0.5 else 'Symmetric'
    stats_text = (f'n = {len(df):,}\n'
                  f'Mean: {col_mean:.2f}\n'
                  f'Median: {col_median:.2f}\n'
                  f'Std: {df[column].std():.2f}\n'
                  f'Skew: {skew_val:.2f} ({skew_dir})')

    ax.text(0.98, 0.97, stats_text,
            transform=ax.transAxes,
            fontsize=9, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                      alpha=0.9, edgecolor='#BDBDBD'))

    ax.set_title(title or f'Distribution of {column}', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel(column.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.legend(fontsize=9)

    plt.tight_layout()
    return ax


# ── Demo: call the function for 3 different columns ──
ax1 = plot_distribution_with_stats(
    tips, column='total_bill', group_col='time',
    title='🍽️ Restaurant Bill Distribution by Meal Time',
    palette={'Lunch': '#1565C0', 'Dinner': '#E53935'}
)
plt.show()

ax2 = plot_distribution_with_stats(
    penguins, column='body_mass_g', group_col='species',
    title='🐧 Penguin Body Mass Distribution by Species',
    palette='colorblind'
)
plt.show()

print("✅ Reusable function called twice with 1 line each.")
print("   Same professional styling applied automatically.")
print("   → This is what production-ready code looks like.")

### 📋 Output Preview — Reusable Function
```
Chart 1: Tips total_bill distribution
  - Two filled KDE curves (Lunch=blue, Dinner=red)
  - Red dashed mean line + Blue dash-dot median line
  - Stats box (top-right): n=244, Mean=19.79, Median=17.80, Skew=1.13 (Right-skewed)

Chart 2: Penguin body mass
  - Three filled KDE curves (one per species)
  - Same stats box + reference lines — consistent styling
```
> 🏆 **Interview Story:** *"In my projects, I write reusable plotting functions that accept a DataFrame and column name and produce consistent, publication-quality charts. This makes EDA much faster — I run one line instead of 20 for each variable, and all charts look uniform."*

In [ ]:
# 🔴 Advanced | Reusable correlation dashboard function
def plot_correlation_report(
    df: pd.DataFrame,
    target_col: str,
    title: str = None,
    figsize: tuple = (16, 6)
) -> None:
    """
    Two-panel correlation report:
    - Left: Masked heatmap of all numeric features
    - Right: Horizontal bar chart ranking features by correlation with target

    Parameters
    ----------
    df         : Input DataFrame (numeric columns selected automatically)
    target_col : Target variable for correlation ranking
    title      : Figure suptitle
    figsize    : Figure size
    """
    import numpy as np

    numeric_df = df.select_dtypes(include='number')
    corr       = numeric_df.corr()
    mask       = np.triu(np.ones_like(corr, dtype=bool))

    corr_target = (
        corr[target_col]
        .drop(target_col)
        .sort_values(key=abs, ascending=True)
    )

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # ── Left: Masked heatmap ──
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f',
        cmap='coolwarm', center=0, vmin=-1, vmax=1,
        square=True, linewidths=0.8,
        cbar_kws={'label': 'Pearson r', 'shrink': 0.75},
        ax=axes[0]
    )
    axes[0].set_title('Feature Correlation Matrix (masked)', fontweight='bold', fontsize=11)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

    # ── Right: Feature ranking bar chart ──
    bar_colors = ['#E53935' if abs(v) > 0.6 else '#FB8C00' if abs(v) > 0.3 else '#FDD835'
                  for v in corr_target.values]
    bars = axes[1].barh(
        corr_target.index,
        corr_target.abs().values,
        color=bar_colors,
        edgecolor='white'
    )
    axes[1].bar_label(bars, fmt='%.3f', fontsize=10, fontweight='bold', padding=4)
    axes[1].set_title(f'Feature Correlation with "{target_col}"\n(|Pearson r|)',
                      fontweight='bold', fontsize=11)
    axes[1].set_xlabel('|Pearson r|', fontsize=11)
    axes[1].axvline(0.6, color='#E53935', lw=1.2, linestyle='--', alpha=0.7, label='Strong (0.6)')
    axes[1].axvline(0.3, color='#FB8C00', lw=1.2, linestyle='--', alpha=0.7, label='Moderate (0.3)')
    axes[1].legend(fontsize=9)

    fig.suptitle(title or f'Correlation Report — Target: {target_col}',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    fig.savefig(f'outputs/corr_report_{target_col}.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: outputs/corr_report_{target_col}.png")


# Demo: Run the function on two different datasets/targets
plot_correlation_report(tips, target_col='tip', title='🍽️ Restaurant — Tip Correlation Report')
plot_correlation_report(mpg,  target_col='mpg', title='🚗 Automotive — MPG Correlation Report')

---
<a id='palettes'></a>
## 7. Advanced Color Palettes — Choosing Correctly

### 🧠 The 3 Types of Palettes — A Rule Every Fresher Must Know

| Type | When to Use | Examples |
|------|-------------|---------|
| **Qualitative** | Categorical data with NO order | `Set2`, `deep`, `colorblind`, `tab10` |
| **Sequential** | Ordered data going ONE direction (low→high) | `Blues`, `YlOrRd`, `viridis`, `plasma` |
| **Diverging** | Data centered around a midpoint (+/−) | `coolwarm`, `RdBu`, `vlag`, `PiYG` |

> 🎯 **Interview Rule:** *"I always use qualitative palettes for categories (like species, day), sequential for ordered/numeric heatmaps (like sales amount), and diverging for correlations (which go from −1 to +1)."*

In [ ]:
# 🟡 Intermediate | Palette type comparison — see ALL three types
fig, axes = plt.subplots(3, 3, figsize=(15, 10))

# ── ROW 1: Qualitative palettes ──
sns.countplot(data=tips, x="day", order=["Thur","Fri","Sat","Sun"],
              hue="day", palette="Set2", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("palette='Set2'
Good general purpose", fontweight="bold", fontsize=9)
axes[0, 0].set_xlabel("Day"); axes[0, 0].set_ylabel("Count")

sns.countplot(data=tips, x="day", order=["Thur","Fri","Sat","Sun"],
              hue="day", palette="colorblind", legend=False, ax=axes[0, 1])
axes[0, 1].set_title("palette='colorblind'
Accessibility-friendly ✅", fontweight="bold", fontsize=9)
axes[0, 1].set_xlabel("Day"); axes[0, 1].set_ylabel("Count")

sns.countplot(data=tips, x="day", order=["Thur","Fri","Sat","Sun"],
              hue="day", palette="tab10", legend=False, ax=axes[0, 2])
axes[0, 2].set_title("palette='tab10'
High-contrast, many groups", fontweight="bold", fontsize=9)
axes[0, 2].set_xlabel("Day"); axes[0, 2].set_ylabel("Count")

# ── ROW 2: Sequential palettes ──
sns.histplot(data=tips, x="total_bill", color=sns.color_palette("Blues", 1)[0],
             bins=15, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("palette='Blues'
Low=light, High=dark", fontweight="bold", fontsize=9)
axes[1, 0].set_xlabel("Total Bill ($)")

sns.histplot(data=tips, x="total_bill", color=sns.color_palette("YlOrRd", 1)[0],
             bins=15, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("palette='YlOrRd'
Yellow→Orange→Red", fontweight="bold", fontsize=9)
axes[1, 1].set_xlabel("Total Bill ($)")

sns.histplot(data=tips, x="total_bill", color=sns.color_palette("viridis", 1)[0],
             bins=15, kde=True, ax=axes[1, 2])
axes[1, 2].set_title("palette='viridis'
Colorblind-safe ✅", fontweight="bold", fontsize=9)
axes[1, 2].set_xlabel("Total Bill ($)")

# ── ROW 3: Diverging palettes ──
corr_tips = tips.select_dtypes(include="number").corr()
mask_tips  = np.triu(np.ones_like(corr_tips, dtype=bool))

sns.heatmap(corr_tips, mask=mask_tips, cmap="coolwarm", center=0,
            annot=True, fmt=".1f", square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.7}, ax=axes[2, 0])
axes[2, 0].set_title("cmap='coolwarm'
Blue=neg, Red=pos (most common)", fontweight="bold", fontsize=9)
axes[2, 0].set_xticklabels(axes[2, 0].get_xticklabels(), rotation=30, ha="right", fontsize=7)

sns.heatmap(corr_tips, mask=mask_tips, cmap="RdBu_r", center=0,
            annot=True, fmt=".1f", square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.7}, ax=axes[2, 1])
axes[2, 1].set_title("cmap='RdBu_r'
Red=neg, Blue=pos (reversed)", fontweight="bold", fontsize=9)
axes[2, 1].set_xticklabels(axes[2, 1].get_xticklabels(), rotation=30, ha="right", fontsize=7)

sns.heatmap(corr_tips, mask=mask_tips, cmap="vlag", center=0,
            annot=True, fmt=".1f", square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.7}, ax=axes[2, 2])
axes[2, 2].set_title("cmap='vlag'
Muted — good for publications", fontweight="bold", fontsize=9)
axes[2, 2].set_xticklabels(axes[2, 2].get_xticklabels(), rotation=30, ha="right", fontsize=7)

fig.suptitle("🎨 Palette Selection Guide — Qualitative | Sequential | Diverging",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 🔴 Advanced | Custom palette — brand color matching
# In real jobs, you often need to match company brand colors.
# Custom palettes let you use exact hex codes for any chart.

# ── Custom brand-style palette ──
brand_palette = {
    'Thur': '#1565C0',   # Brand blue
    'Fri':  '#2E7D32',   # Brand green
    'Sat':  '#E65100',   # Brand orange
    'Sun':  '#6A1B9A',   # Brand purple
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Custom dict palette (maps category → exact color)
sns.countplot(
    data=tips, x='day', order=['Thur','Fri','Sat','Sun'],
    hue='day', palette=brand_palette,
    legend=False, ax=axes[0]
)
axes[0].set_title('Custom Dict Palette\n(Exact brand hex colors)', fontweight='bold')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Count')

# Right: sns.color_palette with custom hex list
custom_seq = sns.color_palette(['#E8F5E9', '#66BB6A', '#2E7D32', '#1B5E20'])
sns.barplot(
    data=tips, x='day', y='total_bill',
    order=['Thur', 'Fri', 'Sat', 'Sun'],
    hue='day',
    palette=custom_seq,
    estimator='mean', errorbar=None,
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Custom Sequential Hex List\n(4 shades of brand green)',
                  fontweight='bold')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Avg Total Bill ($)')

fig.suptitle('🎨 Custom Palettes — Brand Colors in Seaborn',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("💡 PALETTE CREATION METHODS:")
print("   1. Dict palette:  {'cat1': '#hex1', 'cat2': '#hex2'}")
print("   2. List palette:  sns.color_palette(['#hex1', '#hex2', ...])")
print("   3. Named palette: palette='coolwarm' or palette='Set2'")
print("   4. Seaborn light: sns.light_palette('#2E7D32', n_colors=5)")

---
<a id='dashboard'></a>
## 8. Business Dashboard — Putting Advanced Techniques Together

In [ ]:
# 🔴 Advanced | Full advanced dashboard combining all techniques
# Demonstrates: FacetGrid, twinx, annotations, custom palette, reusable style
# Business Goal: Comprehensive Penguin Research Report

fig = plt.figure(figsize=(18, 13))

# Define subplot layout: 2 rows, 3 cols
ax_top_left  = fig.add_subplot(2, 3, 1)
ax_top_mid   = fig.add_subplot(2, 3, 2)
ax_top_right = fig.add_subplot(2, 3, 3)
ax_bot_left  = fig.add_subplot(2, 3, 4)
ax_bot_mid   = fig.add_subplot(2, 3, 5)
ax_bot_right = fig.add_subplot(2, 3, 6)

species_palette = {'Adelie': '#E53935', 'Chinstrap': '#1565C0', 'Gentoo': '#2E7D32'}

# ── Panel 1: Bill length distribution by species ──
sns.kdeplot(data=penguins, x='bill_length_mm', hue='species',
            fill=True, alpha=0.45, palette=species_palette,
            common_norm=False, ax=ax_top_left)
ax_top_left.set_title('① Bill Length Distribution', fontweight='bold', fontsize=10)
ax_top_left.set_xlabel('Bill Length (mm)')
ax_top_left.legend(title='Species', fontsize=8)

# ── Panel 2: Body mass by island (boxplot) ──
sns.boxplot(data=penguins, x='island', y='body_mass_g',
            hue='island', palette='Set2', legend=False,
            width=0.5, ax=ax_top_mid)
ax_top_mid.set_title('② Body Mass by Island', fontweight='bold', fontsize=10)
ax_top_mid.set_xlabel('Island')
ax_top_mid.set_ylabel('Body Mass (g)')

# ── Panel 3: Scatter — flipper vs mass annotated ──
sns.scatterplot(data=penguins, x='flipper_length_mm', y='body_mass_g',
                hue='species', palette=species_palette,
                alpha=0.65, s=45, ax=ax_top_right)
# Annotate Gentoo cluster
ax_top_right.annotate(
    'Gentoo
(largest)',
    xy=(215, 5500), xytext=(200, 3800),
    arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5),
    fontsize=8, color='#2E7D32', fontweight='bold',
    bbox=dict(boxstyle='round', fc='#E8F5E9', alpha=0.9, ec='#2E7D32')
)
ax_top_right.set_title('③ Flipper vs Body Mass', fontweight='bold', fontsize=10)
ax_top_right.set_xlabel('Flipper Length (mm)')
ax_top_right.set_ylabel('Body Mass (g)')
ax_top_right.legend(title='Species', fontsize=8)

# ── Panel 4: Masked correlation heatmap ──
peng_corr = penguins.select_dtypes(include='number').corr()
peng_mask = np.triu(np.ones_like(peng_corr, dtype=bool))
sns.heatmap(peng_corr, mask=peng_mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            linewidths=0.8, cbar_kws={'shrink': 0.7},
            ax=ax_bot_left)
ax_bot_left.set_title('④ Feature Correlation (masked)', fontweight='bold', fontsize=10)
ax_bot_left.set_xticklabels(ax_bot_left.get_xticklabels(), rotation=30, ha='right', fontsize=8)

# ── Panel 5: Species count by sex ──
sns.countplot(data=penguins, x='species', hue='sex',
              palette={'Male': '#1565C0', 'Female': '#C62828'},
              order=['Adelie', 'Chinstrap', 'Gentoo'],
              ax=ax_bot_mid)
ax_bot_mid.set_title('⑤ Species Count by Sex', fontweight='bold', fontsize=10)
ax_bot_mid.set_xlabel('Species')
ax_bot_mid.set_ylabel('Count')
ax_bot_mid.legend(title='Sex', fontsize=8)

# ── Panel 6: Feature correlation ranking with target ──
corr_target = peng_corr['body_mass_g'].drop('body_mass_g').abs().sort_values(ascending=True)
bar_colors_p = pd.Series(corr_target.values)
bar_colors_p = bar_colors_p.map(lambda v: '#E53935' if v > 0.6 else '#FB8C00').tolist()
b6 = ax_bot_right.barh(corr_target.index, corr_target.values,
                        color=bar_colors_p, edgecolor='white')
ax_bot_right.bar_label(b6, fmt='%.3f', fontsize=9, fontweight='bold', padding=3)
ax_bot_right.set_title('⑥ Correlation with Body Mass', fontweight='bold', fontsize=10)
ax_bot_right.set_xlabel('|Pearson r|')

fig.suptitle('🐧 Penguin Research — Advanced EDA Dashboard',
             fontsize=15, fontweight='bold', y=1.01)
fig.text(0.99, 0.01, 'Data: Palmer Penguins | Seaborn Masterclass NB5',
         ha='right', fontsize=8, color='gray', style='italic')

plt.tight_layout()
fig.savefig('outputs/penguin_advanced_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Dashboard saved: outputs/penguin_advanced_dashboard.png")
print()
print("📊 KEY FINDINGS:")
print(f"   Flipper length ↔ body mass: r = {peng_corr.loc['flipper_length_mm','body_mass_g']:.3f} (strongest predictor)")
print(f"   Gentoo clearly largest: heaviest + longest flipper")
print(f"   Bill depth ↔ bill length: r = {peng_corr.loc['bill_depth_mm','bill_length_mm']:.3f} (weak negative — Simpson's Paradox!)")

---
<a id='best-practices'></a>
## 9. Best Practices

| Category | Best Practice |
|----------|--------------|
| **FacetGrid** | Always use `map_dataframe()` over `map()` — supports all kwargs including `hue=` |
| **FacetGrid** | Use `g.axes_dict` to access individual panels without loops |
| **FacetGrid** | Set `margin_titles=True` for cleaner 2D (row+col) grids |
| **PairGrid** | Always use `map_diag`, `map_upper`, `map_lower` separately — never default to pairplot when you need custom plots |
| **Annotations** | Find key points with `df.idxmax()` / `df.idxmin()` before plotting — never hardcode coordinates |
| **Annotations** | Use `bbox=dict(...)` to box annotations — unboxed text blends with data |
| **twinx** | Combine legends manually: `h1+h2, l1+l2 = ax1.get_legend_handles_labels()` |
| **twinx** | Color-match Y-axis labels and tick marks to each line — critical for readability |
| **Reusable functions** | Always `return ax` — lets callers add further customization |
| **Reusable functions** | Use type hints: `def func(df: pd.DataFrame, col: str) -> plt.Axes:` |
| **Palettes** | Use `'colorblind'` for presentations to diverse audiences |
| **Palettes** | Match palette type to data type: qualitative/sequential/diverging |

---
<a id='common-mistakes'></a>
## 10. Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| Using `g.map()` instead of `g.map_dataframe()` | Can't pass kwargs like `hue=` | Always use `map_dataframe()` |
| Accessing FacetGrid panels with loops | Violates no-loop rule; use `g.axes_dict` instead | `ax = g.axes_dict['value']` |
| Hardcoding annotation coordinates | Chart breaks when data changes | Use `df.idxmax()` to find points dynamically |
| Using 3+ Y-axes | Incomprehensible chart | Use stacked subplots instead |
| Not matching tick colors in twinx | Right Y-axis is confusing — which line is which? | `ax2.tick_params(labelcolor='red')` |
| Missing `return ax` in reusable functions | Caller can't customize further | Always `return ax` |
| Using qualitative palette for numeric heatmap | Colors don't convey magnitude | Use sequential or diverging cmap |
| Using diverging palette for categories | Implies some are "negative" vs "positive" | Use qualitative palette |

---
<a id='interview-notes'></a>
## 11. 🎤 Interview Notes

---
**Q: What is FacetGrid and when would you use it directly instead of relplot?**
> *"FacetGrid is the underlying class that all figure-level functions (relplot, displot, catplot) use internally. I'd use it directly when I need to combine multiple plot types in one faceted layout — for example, KDE + rug plot on the same panels, or when I need `map_dataframe()` to pass complex kwargs like multiple hue parameters that relplot doesn't support natively."*

---
**Q: What is the difference between map() and map_dataframe() in FacetGrid?**
> *"`map()` passes individual Series (columns) to the function — it works for simple cases but can't pass kwargs like `hue=` or `order=`. `map_dataframe()` passes the full filtered DataFrame slice plus any keyword arguments — it's more flexible and is the preferred method for any non-trivial mapping."*

---
**Q: How do you create a dual Y-axis chart in Matplotlib/Seaborn?**
> *"`ax2 = ax1.twinx()` creates a second Axes that shares the X-axis but has an independent Y-axis. I then plot on ax1 and ax2 separately, set different y-labels in different colors, use `tick_params(labelcolor=)` to color the tick labels, and combine both legends manually using `get_legend_handles_labels()`."*

---
**Q: Why would you write reusable plotting functions instead of writing charts from scratch each time?**
> *"Reusable functions ensure consistency across charts, reduce code duplication, and make the codebase maintainable. If the company style guide changes, I update one function instead of 50 chart blocks. They also make it easier to onboard teammates — they call one function with clear parameters instead of understanding 20 lines of chart setup. I always return the Axes object so callers can further customize if needed."*

---
<a id='summary'></a>
## 12. Summary & Key Takeaways

| Concept | One-Line Summary |
|---------|-----------------|
| `FacetGrid` | Low-level faceting engine — use `map_dataframe()`, access panels via `axes_dict` |
| `PairGrid` | Assign different plot types to diagonal, upper, lower triangles independently |
| `ax.annotate()` | Add arrows + callout boxes to specific data points — makes charts self-explaining |
| `ax.twinx()` | Two Y-axes sharing one X — for metrics with different scales |
| Reusable functions | `def plot_X(df, col) -> ax:` — production pattern every fresher should know |
| Palette selection | Qualitative=categories, Sequential=ordered, Diverging=centered around 0 |

### 🏆 The 3 Patterns to Memorize

```python
# 1. FacetGrid with map_dataframe (more powerful than relplot)
g = sns.FacetGrid(df, col='cat', hue='cat2', height=4)
g.map_dataframe(sns.scatterplot, x='x', y='y')
g.add_legend()

# 2. PairGrid with 3 different plots
g = sns.PairGrid(df, hue='species')
g.map_diag(sns.histplot, fill=True)
g.map_upper(sns.scatterplot, alpha=0.5)
g.map_lower(sns.kdeplot, fill=True)

# 3. Reusable function pattern
def plot_dist(df, col, title) -> plt.Axes:
    fig, ax = plt.subplots()
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(title)
    return ax
```

---

<a id='practice'></a>
## 13. 📝 Practice Questions

---
### 🟢 Easy (1–5)

**Q1.** Create a `FacetGrid` from `tips` with `col='day'`, `col_order=['Thur','Fri','Sat','Sun']`. Use `map_dataframe(sns.histplot, x='tip', bins=12)`. Add axis labels and panel titles.

**Q2.** Create a `PairGrid` from `penguins` using `vars=['bill_length_mm','flipper_length_mm','body_mass_g']` and `hue='species'`. Use `map_diag(sns.histplot)` and `map_upper(sns.scatterplot)`. Add legend.

**Q3.** Create a scatterplot of `mpg` vs `horsepower` from the `mpg` dataset. Annotate the point with the highest mpg using `ax.annotate()` with an arrow. Use `df.idxmax()` to find it.

**Q4.** Create a `twinx` chart for the `yearly_flights` DataFrame: bars for total passengers (left Y), line for YoY growth % (right Y). Color left Y blue, right Y red.

**Q5.** Write a reusable function `plot_boxplot(df, x, y, title)` that creates a styled boxplot and returns the Axes. Call it for `tips` (x='day', y='tip') and `penguins` (x='species', y='body_mass_g').

---
### 🟡 Medium (6–11)

**Q6.** Create a 2×2 `FacetGrid` from `tips` with `col='time'` and `row='smoker'`. In each panel, draw both a KDE and a rug plot using two `map_dataframe()` calls. Add median vertical lines on each panel using `g.axes_dict`.

**Q7.** Build a `PairGrid` for the `mpg` numeric features. Use:
- Diagonal: `sns.histplot(kde=True)`
- Upper: `sns.scatterplot(alpha=0.3)`
- Lower: `sns.regplot(ci=None, scatter_kws={'alpha':0.2})`
Which pair of features shows the strongest linear relationship?

**Q8.** Create a scatter of `total_bill` vs `tip_pct` (tip as % of bill). Annotate:
- The table with the highest tip % (biggest tipper)
- The table with the lowest tip % (stingiest)
- Any outlier above 40% tip
Use `ax.annotate()` with a curved arrow (`connectionstyle='arc3,rad=0.3'`).

**Q9.** Create a `twinx` chart using `mpg` grouped by `model_year`:
- Left Y: average `mpg` (bars)
- Right Y: average `horsepower` (line with markers)
Annotate the year with the biggest mpg improvement with an arrow callout.

**Q10.** Create a `FacetGrid` for `exercise` with `col='kind'` (rest/walking/running). In each panel use `map_dataframe(sns.lineplot, x='time', y='pulse', hue='diet')`. Add legend to all panels.

**Q11.** Write a reusable function `plot_corr_heatmap(df, title)` that:
- Selects numeric columns automatically
- Applies upper triangle mask
- Uses coolwarm, center=0, annot=True
- Saves to `outputs/{title}.png`
- Returns the Axes
Call it for `tips`, `penguins`, and `mpg`.

---
### 🔴 Hard (12–15)

**Q12. 🏗️ Advanced FacetGrid Layout.**
Using `penguins`, create a `FacetGrid` with `col='species'` and `row='sex'`. In each panel:
1. Draw a scatter of `bill_length_mm` vs `body_mass_g`
2. Add a regression line using `map_dataframe` + `sns.regplot` (scatter=False, ci=None)
3. Draw the scatter using `map_dataframe` + `sns.scatterplot`
4. Access `g.axes_dict` to add a custom title to each panel showing n= (count for that group)
Write 3 biological insights about how species and sex interact.

**Q13. 🔍 Full PairGrid Report.**
Build the most information-dense PairGrid possible for `mpg` numeric features:
- Diagonal: KDE with fill, one color per group (color by `origin` using `hue=`)
- Upper: Scatter colored by origin
- Lower: KDE contour colored by origin
- Legend showing all 3 origins
Calculate and print the F-ratio (between-group variance / within-group variance using Pandas `.groupby().var()`) for each feature — which feature best separates USA/Europe/Japan?

**Q14. 🎨 Full Annotation Dashboard.**
Using `flights`, build an annotated time series dashboard:
1. Line chart of total passengers per year
2. Annotate: (a) fastest growth year, (b) slowest growth year, (c) the year WWII ended (1945 is not in data — annotate 1949 as first post-war data point), (d) total passengers in the final year
3. Add shaded regions for decade boundaries (1949–1959)
4. Use `twinx` to add YoY growth % as a secondary line
5. Export to PNG

**Q15. 🏆 Reusable EDA Pipeline.**
Write a function `run_eda(df, target_col, dataset_name)` that:
1. Prints shape, dtypes, missing values (using Pandas)
2. Creates a masked correlation heatmap (saved as PNG)
3. Creates a feature correlation ranking bar chart
4. Creates a pairplot with `hue=` set to the highest-correlated categorical column (find it with Pandas)
5. Returns a dict with keys: `{'corr_matrix': ..., 'top_feature': ..., 'missing_count': ...}`

Call it on `tips` (target='tip') and `penguins` (target='body_mass_g').

---
### 🔥 Predict the Output Challenge

```python
g = sns.FacetGrid(tips, col='day', col_wrap=2,
                  height=3, aspect=1.1, hue='sex',
                  palette={'Male': 'blue', 'Female': 'red'})
g.map_dataframe(sns.scatterplot, x='total_bill', y='tip', alpha=0.6)
g.add_legend()
g.set_titles(col_template='Day: {col_name}')
ax_sat = g.axes_dict['Sat']
ax_sat.axhline(y=5, color='green', linestyle='--', label='$5 tip line')
plt.show()
```

❓ Questions (answer before running):
1. How many panels does this create and in what layout?
2. What does `col_wrap=2` do?
3. Can you use `ax=` when calling FacetGrid? Why or why not?
4. What does `g.axes_dict['Sat']` return?
5. The green line only appears on Saturday's panel. How would you add it to ALL panels without a loop?

---

<div style="background: linear-gradient(135deg, #1a0533 0%, #2d1b69 100%); padding: 30px; border-radius: 12px; text-align: center; margin-top: 40px;">
  <h2 style="color: #ce93d8; font-size: 1.8em;">🎉 Notebook 5 Complete!</h2>
  <p style="color: #ede7f6; font-size: 1.1em;">You now know FacetGrid, PairGrid, annotations, twin axes, and reusable patterns.</p>
  <p style="color: #ccc; font-size: 1em;">These are the techniques that <strong style='color:#ce93d8'>set senior freshers apart</strong> from average candidates.</p>
  <hr style="border: 1px solid #ce93d8; margin: 15px auto; width: 60%;">
  <p style="color: #ccc;">📚 Next: <strong style='color:#ede7f6'>Notebook 6</strong> — Real-World Case Studies</p>
  <p style="color: #aaa; font-size: 0.9em;">Restaurant Analytics · Titanic Survival · Diamond Pricing · Aviation Trends</p>
</div>